# Accordion Performance Benchmarks

Measure round-trip latency for getting and setting channel values via the Accordion hardware wrapper.

| Test | Description |
|------|-------------|
| **A1** | `set_value()` — single channel |
| **A2** | `set_values()` — multiple channels in one call |
| **B1** | `get_value()` — single channel |
| **B2** | `get_values()` — multiple channels in one call |

In [13]:
import time
import statistics
import accordion as acc

## Connect to hardware

In [14]:
HOST = "agentdemo.local"
acc.attach("perf-bench", HOST)
print(f"Connected: {acc.get_connected()}")
print(f"ID: {acc.get_identification()}")

Connected: True
ID: agentdemo
Name=Control Module 32ch, ID=ESH10000023, S/N=A002274, Rev=4
Name=AGENT Q2 Base, ID=ESH10000158, S/N=A002814, Rev=4
Name=I2Cx4 Long Range Module, ID=ESH10000239, S/N=A000000, Rev=2
Name=MPIO-96 SPI Module, ID=ESH10000568, S/N=A000000, Rev=0
Name=AGENT Q2 TOP, ID=ESH10000183, S/N=A002838, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002612, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002613, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002611, Rev=6
Name=N-Module 6xIDC, ID=ESH10000355, S/N=A002614, Rev=6
Name=N5 TOP, ID=ESH10000359, S/N=A002911, Rev=2
Name=PSU1, ID=ESH10000533, S/N=A002727, Rev=3
Name=Active load 2ch, ID=ESH10000536, S/N=A002848, Rev=2



## Channel setup
Target channels: `MPIO00` – `MPIO07`

In [15]:
single_channel = "MPIO00"
multi_channels = [f"MPIO0{i}" for i in range(8)]  # MPIO00 – MPIO07

# Verify channels exist on the hardware
available = set(acc.get_channel_names())
for ch in multi_channels:
    if ch not in available:
        print(f"WARNING: '{ch}' not found in available channels!")
    else:
        print(f"  OK: {ch}")

print(f"\nSingle-channel target : {single_channel}")
print(f"Multi-channel targets : {multi_channels}")

  OK: MPIO00
  OK: MPIO01
  OK: MPIO02
  OK: MPIO03
  OK: MPIO04
  OK: MPIO05
  OK: MPIO06
  OK: MPIO07

Single-channel target : MPIO00
Multi-channel targets : ['MPIO00', 'MPIO01', 'MPIO02', 'MPIO03', 'MPIO04', 'MPIO05', 'MPIO06', 'MPIO07']


## Benchmark helper

In [16]:
def benchmark(func, iterations=100, label=""):
    """Run *func* repeatedly and return timing statistics (in ms)."""
    times = []
    for _ in range(iterations):
        t0 = time.perf_counter()
        func()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)  # ms

    mean   = statistics.mean(times)
    median = statistics.median(times)
    stdev  = statistics.stdev(times) if len(times) > 1 else 0
    lo, hi = min(times), max(times)

    print(f"\n--- {label} ({iterations} iterations) ---")
    print(f"  Mean   : {mean:8.3f} ms")
    print(f"  Median : {median:8.3f} ms")
    print(f"  Stdev  : {stdev:8.3f} ms")
    print(f"  Min    : {lo:8.3f} ms")
    print(f"  Max    : {hi:8.3f} ms")

    return {"label": label, "mean": mean, "median": median,
            "stdev": stdev, "min": lo, "max": hi, "times": times}

## A1 — `set_value()` single channel

In [17]:
ITERATIONS = 200

# Configure channels as outputs for set_value tests
for ch in multi_channels:
    acc.configure_channelas_output(ch)
print("Channels configured as outputs")

# Read current value so we can write it back (non-destructive)
original = acc.get_value(single_channel)

result_set_single = benchmark(
    lambda: acc.set_value(single_channel, original),
    iterations=ITERATIONS,
    label=f"set_value  (1 ch: {single_channel})"
)

Channels configured as outputs

--- set_value  (1 ch: MPIO00) (200 iterations) ---
  Mean   :   55.772 ms
  Median :   13.735 ms
  Stdev  :   84.202 ms
  Min    :    6.722 ms
  Max    :  232.323 ms


## A2 — `set_values()` multiple channels

In [18]:
originals = list(acc.get_values(multi_channels))

result_set_multi = benchmark(
    lambda: acc.set_values(multi_channels, originals),
    iterations=ITERATIONS,
    label=f"set_values ({len(multi_channels)} ch)"
)

Exception: GetChannelFromNetName: [MPIO00] no such net name

   at System.Reflection.MethodBaseInvoker.InvokeWithFewArgs(Object obj, BindingFlags invokeAttr, Binder binder, Object[] parameters, CultureInfo culture)
   at ProtocolServerImpl.Reflection.ReflectionStub.FunctionInvocation(ReflectionCall call)
   at ProtocolServerImpl.Reflection.ReflectionStub.Server_ReflectionCallReceived(Guid g, ReflectionCall call)
   at ProtocolServerImpl.Reflection.ReflectionStub.SendReceive(ReflectionCall c) in C:\dev\source\accordionpilot\submodules\protocolserver\ProtocolServer\Reflection\ReflectionStub.cs:line 418
   at ProtocolServerImpl.Proxies.ResourceValuesProxy.SetValues(String[] names, String[] values) in C:\dev\source\accordionpilot\submodules\protocolserver\ProtocolServer\Proxies\ResourceValuesProxy.cs:line 39
   at ProtocolServerImpl.Impl.HardwareManager.SetValues(String[] names, String[] values) in C:\dev\source\accordionpilot\submodules\protocolserver\ProtocolServer\Impl\HardwareManag

## B1 — `get_value()` single channel

In [19]:
# Configure channels as inputs for get_value tests
for ch in multi_channels:
    acc.configure_channelas_input(ch)
print("Channels configured as inputs")

result_get_single = benchmark(
    lambda: acc.get_value(single_channel),
    iterations=ITERATIONS,
    label=f"get_value  (1 ch: {single_channel})"
)

Channels configured as inputs

--- get_value  (1 ch: MPIO00) (200 iterations) ---
  Mean   :   53.756 ms
  Median :   13.858 ms
  Stdev  :   83.156 ms
  Min    :    6.754 ms
  Max    :  236.700 ms


## B2 — `get_values()` multiple channels

In [20]:
result_get_multi = benchmark(
    lambda: acc.get_values(multi_channels),
    iterations=ITERATIONS,
    label=f"get_values ({len(multi_channels)} ch)"
)


--- get_values (8 ch) (200 iterations) ---
  Mean   :   84.000 ms
  Median :   21.144 ms
  Stdev  :   96.798 ms
  Min    :    9.973 ms
  Max    :  243.519 ms


## Summary

In [12]:
results = [result_set_single, result_set_multi, result_get_single, result_get_multi]

print(f"{'Test':<35} {'Mean (ms)':>10} {'Median (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 80)
for r in results:
    print(f"{r['label']:<35} {r['mean']:>10.3f} {r['median']:>12.3f} {r['min']:>10.3f} {r['max']:>10.3f}")

NameError: name 'result_set_multi' is not defined

## Disconnect

In [ ]:
acc.detach()
print("Detached.")